# Chapter 36: Stereo and RGBD SLAM

<a href="../lite/lab/index.html?path=ch36_stereo_rgbd.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

Stereo cameras solve the scale problem by using two eyes, just like you do.
RGBD cameras cheat: they project infrared dots and measure depth directly.
Both give you dense depth, but both fail in different, fascinating ways.

In this chapter we implement:
- **Stereo depth recovery** from disparity: $Z = fB / d$
- **Epipolar geometry** of rectified stereo pairs
- **Accuracy analysis** showing when each sensor type works and when it breaks
- **Failure modes** specific to stereo and RGBD systems

```{admonition} What you will build
:class: tip

- Recover depth from stereo disparity: Z = f * B / d
- Show that stereo depth accuracy degrades quadratically with distance
- Compare stereo and RGBD sensors for accuracy, range, and failure modes
- Build a stereo depth estimator from simulated left and right camera images

**Real world application:** Stereo cameras power autonomous vehicles (Subaru EyeSight). RGBD cameras power indoor robots (iRobot). After this chapter, you will know the tradeoffs between these two approaches to dense depth.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **ORB-SLAM3 (stereo / RGBD mode)** | Full SLAM with stereo or depth cameras |
| **RTAB-Map (ROS 2)** | Supports stereo and RGBD cameras with loop closure |
| **RealSense SDK** | Intel RealSense RGBD camera driver and processing |
| **ZED SDK** | Stereolabs ZED stereo camera with built-in depth and tracking |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## 36.1 Stereo Depth Recovery

A rectified stereo pair has two cameras with parallel optical axes, separated by
a **baseline** $B$. A 3D point at depth $Z$ projects to pixel $u_L$ in the left
camera and $u_R$ in the right camera. The **disparity** is:

$$d = u_L - u_R = \frac{f \cdot B}{Z}$$

Inverting: $Z = f \cdot B / d$. Larger disparity means closer objects.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(42)
n_pts = 25                       # number of 3D points
focal = 500.0                    # focal length (px)
baseline_stereo = 0.12           # baseline (m), typical for stereo camera
cx, cy = 320.0, 240.0            # principal point
depth_range = (1.0, 15.0)        # depth range of scene (m)
# ─────────────────────────────────────────────────────────────────────────────

# Generate 3D points
pts_3d = np.column_stack([
    np.random.uniform(-3, 3, n_pts),
    np.random.uniform(-2, 2, n_pts),
    np.random.uniform(depth_range[0], depth_range[1], n_pts)
])

# Project to left camera (at origin)
u_left = focal * pts_3d[:, 0] / pts_3d[:, 2] + cx
v_left = focal * pts_3d[:, 1] / pts_3d[:, 2] + cy

# Project to right camera (shifted by baseline along x)
u_right = focal * (pts_3d[:, 0] - baseline_stereo) / pts_3d[:, 2] + cx
v_right = v_left  # same row in rectified stereo

# Compute disparity
disparity = u_left - u_right

# Recover depth from disparity
depth_recovered = focal * baseline_stereo / disparity

# Comparison
depth_error = np.abs(depth_recovered - pts_3d[:, 2])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
sc = ax.scatter(u_left, v_left, c=pts_3d[:, 2], cmap='viridis', s=60,
                edgecolors='k', linewidth=0.5)
plt.colorbar(sc, ax=ax, label='True depth (m)')
ax.set_xlim(0, 640); ax.set_ylim(480, 0)
ax.set_title('Left camera image', fontsize=12)
ax.set_xlabel('u (px)'); ax.set_ylabel('v (px)')

ax = axes[1]
ax.bar(range(n_pts), disparity, color='steelblue', alpha=0.7)
ax.set_xlabel('Point index', fontsize=11)
ax.set_ylabel('Disparity (px)', fontsize=11)
ax.set_title('Disparity (close = large, far = small)', fontsize=12)

ax = axes[2]
ax.scatter(pts_3d[:, 2], depth_recovered, c='forestgreen', s=50,
           edgecolors='k', linewidth=0.5)
z_range = np.linspace(depth_range[0], depth_range[1], 50)
ax.plot(z_range, z_range, 'tomato', linewidth=2, linestyle='--', label='Perfect recovery')
ax.set_xlabel('True depth (m)', fontsize=11)
ax.set_ylabel('Recovered depth (m)', fontsize=11)
ax.set_title('Depth recovery (no noise)', fontsize=12)
ax.legend()

plt.suptitle(f'Stereo Depth: f={focal}px, B={baseline_stereo}m',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Max depth recovery error (noiseless): {np.max(depth_error):.2e} m')
print(f'This confirms Z = fB/d is exact when disparity is known perfectly.')

In [ ]:
# Now add pixel noise and see how depth accuracy degrades
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
pixel_noise_std = 0.5            # sub-pixel noise in disparity
# ─────────────────────────────────────────────────────────────────────────────

disparity_noisy = disparity + np.random.normal(0, pixel_noise_std, n_pts)
depth_noisy = focal * baseline_stereo / disparity_noisy
depth_error_noisy = np.abs(depth_noisy - pts_3d[:, 2])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(pts_3d[:, 2], depth_noisy, c='steelblue', s=50,
           edgecolors='k', linewidth=0.5)
ax.plot(z_range, z_range, 'tomato', linewidth=2, linestyle='--', label='Perfect')
ax.set_xlabel('True depth (m)', fontsize=12)
ax.set_ylabel('Recovered depth (m)', fontsize=12)
ax.set_title(f'Depth with {pixel_noise_std}px noise', fontsize=12)
ax.legend()

ax = axes[1]
ax.scatter(pts_3d[:, 2], depth_error_noisy, c='tomato', s=50,
           edgecolors='k', linewidth=0.5)
# Theoretical error: sigma_Z = Z^2/(fB) * sigma_d
z_theory = np.linspace(1, 15, 100)
sigma_Z_theory = z_theory**2 / (focal * baseline_stereo) * pixel_noise_std
ax.plot(z_theory, sigma_Z_theory, 'forestgreen', linewidth=2,
        label='Theory: $Z^2\\sigma_d/(fB)$')
ax.set_xlabel('True depth (m)', fontsize=12)
ax.set_ylabel('Depth error (m)', fontsize=12)
ax.set_title('Error grows quadratically with depth', fontsize=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

**Key result:** Stereo depth error grows as $Z^2$. At twice the distance, the error
is four times worse. This is the fundamental limitation of triangulation based depth.

## 36.2 Stereo Geometry

In a **rectified** stereo pair, epipolar lines are horizontal. This means that
for each point in the left image, its match in the right image is on the **same row**.
The search is 1D, not 2D, which is much faster and more reliable.

The disparity $d$ encodes depth: large disparity for close objects, small for far ones.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(42)
n_viz_pts = 8                    # number of points to visualize
# ─────────────────────────────────────────────────────────────────────────────

# Select a subset of points for clear visualization
idx_viz = np.random.choice(n_pts, n_viz_pts, replace=False)
colors_viz = plt.cm.tab10(np.linspace(0, 1, n_viz_pts))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left image
ax = axes[0]
for k, i in enumerate(idx_viz):
    ax.scatter(u_left[i], v_left[i], c=[colors_viz[k]], s=80, zorder=5, edgecolors='k')
    ax.axhline(v_left[i], color=colors_viz[k], alpha=0.3, linewidth=1)
    ax.annotate(f'{pts_3d[i,2]:.1f}m', (u_left[i]+5, v_left[i]-5), fontsize=9)
ax.set_xlim(0, 640); ax.set_ylim(480, 0)
ax.set_title('Left image', fontsize=13)
ax.set_xlabel('u'); ax.set_ylabel('v')

# Right image
ax = axes[1]
for k, i in enumerate(idx_viz):
    ax.scatter(u_right[i], v_right[i], c=[colors_viz[k]], s=80, zorder=5, edgecolors='k')
    ax.axhline(v_right[i], color=colors_viz[k], alpha=0.3, linewidth=1)
    ax.annotate('', xy=(u_right[i], v_right[i]),
                xytext=(u_left[i], v_left[i]),
                arrowprops=dict(arrowstyle='->', color=colors_viz[k], lw=1.5))
ax.set_xlim(0, 640); ax.set_ylim(480, 0)
ax.set_title('Right image (arrows show disparity)', fontsize=13)
ax.set_xlabel('u'); ax.set_ylabel('v')

plt.suptitle('Rectified stereo: matches lie on the same row',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Depth accuracy vs distance for different baselines
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
baselines_compare = [0.06, 0.12, 0.24, 0.54]   # different baselines (m)
sigma_d = 0.3                    # disparity noise (px)
# ─────────────────────────────────────────────────────────────────────────────

z_range_bl = np.linspace(0.5, 30, 200)
colors_bl = ['steelblue', 'forestgreen', 'orange', 'tomato']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for k, B in enumerate(baselines_compare):
    sigma_Z = z_range_bl**2 / (focal * B) * sigma_d
    ax.plot(z_range_bl, sigma_Z, color=colors_bl[k], linewidth=2,
            label=f'B = {B*100:.0f} cm')
ax.set_xlabel('Depth Z (m)', fontsize=12)
ax.set_ylabel('Depth std $\\sigma_Z$ (m)', fontsize=12)
ax.set_title('Depth accuracy vs distance', fontsize=13)
ax.set_ylim(0, 5)
ax.legend(fontsize=11)

ax = axes[1]
for k, B in enumerate(baselines_compare):
    relative_err = z_range_bl / (focal * B) * sigma_d * 100
    ax.plot(z_range_bl, relative_err, color=colors_bl[k], linewidth=2,
            label=f'B = {B*100:.0f} cm')
ax.set_xlabel('Depth Z (m)', fontsize=12)
ax.set_ylabel('Relative depth error (%)', fontsize=12)
ax.set_title('Relative depth error vs distance', fontsize=13)
ax.set_ylim(0, 30)
ax.axhline(5, color='gray', linestyle=':', alpha=0.5, label='5% threshold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

**Practical rule:** A stereo camera with baseline $B$ produces usable depth
(relative error < 5%) out to roughly $Z_{\max} \approx 0.05 \cdot f \cdot B / \sigma_d$.

## 36.3 Stereo vs RGBD Tradeoffs

| Property | Stereo | RGBD |
|----------|--------|------|
| **Depth method** | Triangulation (disparity) | Time of flight or structured light |
| **Range** | Long (50m+, with large baseline) | Short (~0.5 to 10m) |
| **Outdoor** | Works well | Fails (IR interference from sunlight) |
| **Textureless surfaces** | Fails (no disparity) | Works (active illumination) |
| **Accuracy near** | Good | Excellent |
| **Accuracy far** | Poor (quadratic) | N/A (out of range) |
| **Computation** | Heavy (dense matching) | Light (direct depth) |

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(42)
n_compare = 50                   # number of test points
stereo_baseline = 0.12           # stereo baseline (m)
stereo_focal = 500.0             # stereo focal length (px)
stereo_disp_noise = 0.5          # stereo disparity noise (px)
rgbd_range_max = 8.0             # RGBD max range (m)
rgbd_noise_std = 0.01            # RGBD depth noise (m), constant
# ─────────────────────────────────────────────────────────────────────────────

test_depths = np.linspace(0.5, 20, n_compare)

# Stereo depth error (Monte Carlo)
n_mc = 100
stereo_errors = np.zeros(n_compare)
for i, Z in enumerate(test_depths):
    d_true = stereo_focal * stereo_baseline / Z
    d_noisy = d_true + np.random.normal(0, stereo_disp_noise, n_mc)
    Z_est = stereo_focal * stereo_baseline / d_noisy
    stereo_errors[i] = np.std(Z_est)

# RGBD depth error: constant noise within range, infinite outside
rgbd_errors = np.where(test_depths <= rgbd_range_max, rgbd_noise_std, np.inf)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(test_depths, stereo_errors, 'steelblue', linewidth=2, label='Stereo')
valid_rgbd = test_depths <= rgbd_range_max
ax.plot(test_depths[valid_rgbd], rgbd_errors[valid_rgbd], 'tomato', linewidth=2,
        label='RGBD')
ax.axvline(rgbd_range_max, color='tomato', linestyle=':', alpha=0.5,
           label=f'RGBD max range ({rgbd_range_max}m)')
ax.set_xlabel('True depth (m)', fontsize=12)
ax.set_ylabel('Depth std (m)', fontsize=12)
ax.set_title('Depth accuracy comparison', fontsize=13)
ax.set_ylim(0, 3)
ax.legend(fontsize=10)

# Scenario comparison
ax = axes[1]
scenarios = ['Indoor\n(1-5m)', 'Corridor\n(2-10m)', 'Outdoor\n(5-50m)', 'Textureless\nwall']
stereo_scores = [0.9, 0.7, 0.4, 0.1]
rgbd_scores = [1.0, 0.6, 0.0, 0.9]

x_pos = np.arange(len(scenarios))
w = 0.35
ax.bar(x_pos - w/2, stereo_scores, w, color='steelblue', label='Stereo', alpha=0.8)
ax.bar(x_pos + w/2, rgbd_scores, w, color='tomato', label='RGBD', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(scenarios, fontsize=10)
ax.set_ylabel('Performance (qualitative)', fontsize=11)
ax.set_title('Scenario comparison', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.2)

plt.tight_layout()
plt.show()

**Choosing your sensor:**
- **Indoor robotics** (vacuum cleaners, warehouse robots): RGBD is ideal. Constant
  depth noise, works on textureless walls, cheap.
- **Autonomous driving**: Stereo with wide baseline (50+ cm). Works outdoors, long range.
- **AR/VR headsets**: Stereo with fisheye lenses. Needs to work both indoors and outdoors.

## 36.4 Failure Modes

Both stereo and RGBD have characteristic failure modes. Understanding these
is critical for building robust SLAM systems.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(42)
n_wall = 30                      # points on textureless wall
wall_depth = 5.0                 # wall at this depth
wall_noise = 0.001               # tiny position variation
# ─────────────────────────────────────────────────────────────────────────────

# Failure 1: Stereo on textureless wall
wall_pts = np.column_stack([
    np.random.uniform(-2, 2, n_wall),
    np.random.uniform(-1.5, 1.5, n_wall),
    np.full(n_wall, wall_depth) + np.random.normal(0, wall_noise, n_wall)
])

d_wall = stereo_focal * stereo_baseline / wall_pts[:, 2]
d_wall_noisy = d_wall + np.random.normal(0, 2.0, n_wall)
z_wall_rec = stereo_focal * stereo_baseline / d_wall_noisy

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
ax.scatter(wall_pts[:, 0], wall_pts[:, 1], c='steelblue', s=40, alpha=0.7)
ax.set_title('Textureless wall (real points)', fontsize=12, fontweight='bold')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.text(0.5, 0.05, f'All at Z = {wall_depth:.1f} m', transform=ax.transAxes,
        ha='center', fontsize=11, color='gray')

ax = axes[1]
ax.hist(d_wall, bins=15, color='forestgreen', alpha=0.6, label='True disparity')
ax.hist(d_wall_noisy, bins=15, color='tomato', alpha=0.6, label='Noisy disparity')
ax.set_xlabel('Disparity (px)', fontsize=11)
ax.set_title('Disparity histogram (wall)', fontsize=12)
ax.legend(fontsize=10)

ax = axes[2]
ax.scatter(wall_pts[:, 2], z_wall_rec, c='tomato', s=40)
ax.plot([3, 7], [3, 7], 'gray', linewidth=1, linestyle='--')
ax.set_xlabel('True depth (m)', fontsize=11)
ax.set_ylabel('Recovered depth (m)', fontsize=11)
ax.set_title('Stereo fails on textureless surfaces', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(42)
sunlight_fraction = 0.4          # fraction of RGBD pixels corrupted
# ─────────────────────────────────────────────────────────────────────────────

# Failure 2: RGBD in sunlight (IR interference)
n_rgbd = 50
rgbd_depths = np.random.uniform(1, 8, n_rgbd)
rgbd_measured = rgbd_depths + np.random.normal(0, 0.01, n_rgbd)

corrupted = np.random.random(n_rgbd) < sunlight_fraction
rgbd_measured_sun = rgbd_measured.copy()
rgbd_measured_sun[corrupted] = np.nan

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(rgbd_depths[~corrupted], rgbd_measured_sun[~corrupted],
           c='forestgreen', s=50, label=f'Valid ({(~corrupted).sum()})',
           edgecolors='k', linewidth=0.5)
ax.scatter(rgbd_depths[corrupted], np.full(corrupted.sum(), 0),
           c='tomato', s=50, marker='x', label=f'Dropout ({corrupted.sum()})')
ax.plot([0, 10], [0, 10], 'gray', linewidth=1, linestyle='--')
ax.set_xlabel('True depth (m)', fontsize=12)
ax.set_ylabel('Measured depth (m)', fontsize=12)
ax.set_title('RGBD in sunlight: depth dropout', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Failure 3: Stereo at large distance
ax = axes[1]
far_depths = np.array([5, 10, 20, 30, 40, 50, 60, 80, 100])
far_disp = stereo_focal * stereo_baseline / far_depths

ax.bar(range(len(far_depths)), far_disp, color='steelblue', alpha=0.8)
ax.axhline(1.0, color='tomato', linewidth=2, linestyle='--', label='1 pixel threshold')
ax.axhline(0.5, color='orange', linewidth=2, linestyle=':', label='0.5 pixel (sub-pixel limit)')
ax.set_xticks(range(len(far_depths)))
ax.set_xticklabels([f'{d}m' for d in far_depths], fontsize=9)
ax.set_ylabel('Disparity (px)', fontsize=12)
ax.set_title(f'Disparity at distance (B={stereo_baseline*100:.0f}cm)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

for d, disp in zip(far_depths, far_disp):
    status = 'OK' if disp > 1.0 else ('marginal' if disp > 0.3 else 'UNUSABLE')
    print(f'  Z={d:3d}m  disparity={disp:6.2f}px  [{status}]')

**Failure mode summary:**

| Mode | Sensor | What happens |
|------|--------|-------------|
| Textureless surface | Stereo | Cannot match features; disparity noise dominates |
| Sunlight | RGBD | IR dots washed out; depth becomes NaN |
| Large distance | Stereo | Disparity < 1px; depth error explodes |

## Capstone: Stereo Depth Estimator

Build a complete stereo depth pipeline:
1. Generate a 3D scene with points at various depths
2. Project to left and right cameras
3. Compute disparity for each point
4. Recover depth and build a 3D point cloud
5. Color code by depth error
6. Show how baseline affects the useful depth range

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
np.random.seed(7)
n_scene = 80                     # number of 3D points
f_cap = 500.0                    # focal length
cx_cap, cy_cap = 320.0, 240.0    # principal point
baselines_cap = [0.06, 0.12, 0.30, 0.60]  # baselines to compare
noise_cap = 0.3                  # pixel noise
# ─────────────────────────────────────────────────────────────────────────────

# Generate a structured 3D scene
pts_near = np.column_stack([np.random.uniform(-1, 1, 25),
                            np.random.uniform(-1, 1, 25),
                            np.random.uniform(1, 3, 25)])
pts_mid = np.column_stack([np.random.uniform(-3, 3, 30),
                           np.random.uniform(-2, 2, 30),
                           np.random.uniform(5, 10, 30)])
pts_far = np.column_stack([np.random.uniform(-5, 5, 25),
                           np.random.uniform(-3, 3, 25),
                           np.random.uniform(15, 25, 25)])
pts_scene = np.vstack([pts_near, pts_mid, pts_far])
true_depths_cap = pts_scene[:, 2]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for k, B in enumerate(baselines_cap):
    ax = axes[k // 2, k % 2]
    
    u_L = f_cap * pts_scene[:, 0] / pts_scene[:, 2] + cx_cap
    u_R = f_cap * (pts_scene[:, 0] - B) / pts_scene[:, 2] + cx_cap
    disp_k = (u_L - u_R) + np.random.normal(0, noise_cap, len(pts_scene))
    
    valid_mask = disp_k > 0.1
    z_rec = np.full(len(pts_scene), np.nan)
    z_rec[valid_mask] = f_cap * B / disp_k[valid_mask]
    err_k = np.abs(z_rec - true_depths_cap)
    
    sc = ax.scatter(pts_scene[valid_mask, 0], pts_scene[valid_mask, 2],
                    c=err_k[valid_mask], cmap='RdYlGn_r', s=30, vmin=0, vmax=5,
                    edgecolors='k', linewidth=0.3)
    if (~valid_mask).any():
        ax.scatter(pts_scene[~valid_mask, 0], pts_scene[~valid_mask, 2],
                   c='gray', s=20, marker='x', label='Invalid')
    
    useful_mask = valid_mask & (err_k < 1.0)
    max_useful = np.max(true_depths_cap[useful_mask]) if useful_mask.any() else 0
    ax.axhline(max_useful, color='steelblue', linestyle='--', alpha=0.5)
    ax.set_xlabel('X (m)', fontsize=10)
    ax.set_ylabel('Z / depth (m)', fontsize=10)
    ax.set_title(f'B = {B*100:.0f} cm, useful range ~ {max_useful:.0f} m', fontsize=12)

plt.colorbar(sc, ax=axes, label='Depth error (m)', shrink=0.6)
plt.suptitle('Stereo depth: baseline controls useful range',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3D point cloud reconstruction for the best baseline
B_best = baselines_cap[-1]
u_L = f_cap * pts_scene[:, 0] / pts_scene[:, 2] + cx_cap
u_R = f_cap * (pts_scene[:, 0] - B_best) / pts_scene[:, 2] + cx_cap
v_both = f_cap * pts_scene[:, 1] / pts_scene[:, 2] + cy_cap

disp_best = (u_L - u_R) + np.random.normal(0, noise_cap, len(pts_scene))
valid_best = disp_best > 0.1

z_cloud = f_cap * B_best / disp_best[valid_best]
x_cloud = (u_L[valid_best] - cx_cap) * z_cloud / f_cap
y_cloud = (v_both[valid_best] - cy_cap) * z_cloud / f_cap

err_cloud = np.sqrt((x_cloud - pts_scene[valid_best, 0])**2 +
                     (y_cloud - pts_scene[valid_best, 1])**2 +
                     (z_cloud - pts_scene[valid_best, 2])**2)

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(pts_scene[:, 0], pts_scene[:, 1], pts_scene[:, 2],
            c='steelblue', s=20, alpha=0.6, label='Ground truth')
ax1.scatter(x_cloud, y_cloud, z_cloud, c='tomato', s=20, alpha=0.6,
            label='Recovered')
ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
ax1.set_title(f'3D reconstruction (B={B_best*100:.0f}cm)', fontsize=12)
ax1.legend(fontsize=9)

ax2 = fig.add_subplot(122)
ax2.scatter(pts_scene[valid_best, 2], err_cloud, c=err_cloud, cmap='RdYlGn_r',
            s=40, vmin=0, vmax=3, edgecolors='k', linewidth=0.3)
ax2.set_xlabel('True depth (m)', fontsize=12)
ax2.set_ylabel('3D reconstruction error (m)', fontsize=12)
ax2.set_title('Error vs depth (color = error)', fontsize=12)

plt.tight_layout()
plt.show()

print(f'Baseline: {B_best*100:.0f} cm')
print(f'Valid points: {valid_best.sum()}/{len(pts_scene)}')
print(f'Mean 3D error: {np.mean(err_cloud):.3f} m')
print(f'Median 3D error: {np.median(err_cloud):.3f} m')
print(f'Points with error < 0.5m: {(err_cloud < 0.5).sum()}/{len(err_cloud)}')

**Capstone observations:**
- Wider baselines produce accurate depth at greater distances
- Near points are always well recovered; far points degrade quadratically
- The 3D point cloud clearly shows depth dependent error: near cluster is tight, far cluster is scattered
- In practice, stereo SLAM systems use the depth dependent uncertainty to weight observations properly

---

## Exercises

### Exercise 36.1
Implement a function `max_useful_depth(f, B, sigma_d, max_error)` that returns
the maximum depth at which stereo produces depth with error less than `max_error`.
Plot it for baselines from 5cm to 100cm.

In [ ]:
# Your code here

### Exercise 36.2
Simulate an RGBD camera with **depth dependent noise**: $\sigma_Z = 0.001 Z^2$.
Plot depth error vs distance for this model. At what depth does the RGBD camera
become worse than a stereo camera with B=12cm?

In [ ]:
# Your code here

### Exercise 36.3
Generate a scene where half the points lie on a textureless wall (all at the same
depth) and half are on textured objects (spread across depths). Run stereo depth
recovery with noise. Plot a histogram of depth errors separately for wall points
and textured points.

In [ ]:
# Your code here

### Exercise 36.4
Design a **hybrid** depth sensor: use RGBD for points within 5m and stereo for
points beyond 5m. Implement this logic and show that the hybrid outperforms either
sensor alone across the full depth range (0.5 to 30m).

In [ ]:
# Your code here

### Exercise 36.5
A stereo camera on a self driving car has B=54cm, f=700px. Compute the disparity
for a pedestrian at 5m, 20m, and 50m. At which distance does the disparity drop
below 1 pixel? What does this mean for detection?

In [ ]:
# Your code here